# Práctica 12 — Multicolinealidad, selección de variables y validación

**Curso:** Estadística Aplicada con Python · **Bloque 4:** Modelos y experimentos · **Tiempo estimado:** 60 minutos
**Clase asociada:** `notebooks_clase/clase12_multicolinealidad_seleccion_validacion.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una celda con `# TODO`: complétala.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te dice qué revisar. Si olvidas completar un `TODO`, el mensaje te lo recuerda.
- Los ejercicios con simulación o con datos aleatorios usan tolerancias basadas en el error estándar, así que un resultado correcto no debería fallar por casualidad.
- Las **soluciones** están al final del notebook, en una sección aparte. **No las abras antes de haberlo intentado.**
- Ejecuta las celdas en orden. Todos los datos son sintéticos: *ejemplos ilustrativos*.

---
## Celda 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

# Coeficientes VERDADEROS con los que generamos los datos (solo los conocemos porque son sintéticos)
VERDAD = {"Intercept": -1.8, "antiguedad": -0.03, "gasto": 0.03, "datos_gb": 0.0, "tickets": 0.45, "plan_anual": -0.9}

def generar_clientes(n, rng):
    """Genera n clientes sintéticos (ejemplo ilustrativo) con abandono según VERDAD."""
    antiguedad = np.clip(rng.gamma(2.0, 14, n), 1, 72).round()
    gasto = np.clip(30 + 0.25 * antiguedad + rng.normal(0, 10, n), 10, None).round(1)
    datos_gb = np.clip(0.09 * gasto + rng.normal(0, 0.25, n), 0.1, None).round(2)   # muy correlacionada con gasto
    tickets = rng.poisson(np.exp(0.008 * (gasto - 40)))
    plan_anual = rng.binomial(1, 0.35, n)
    eta = (VERDAD["Intercept"] + VERDAD["antiguedad"] * antiguedad + VERDAD["gasto"] * gasto
           + VERDAD["tickets"] * tickets + VERDAD["plan_anual"] * plan_anual)
    abandona = rng.binomial(1, 1 / (1 + np.exp(-eta)))
    return pd.DataFrame({"abandona": abandona, "antiguedad": antiguedad, "gasto": gasto,
                         "datos_gb": datos_gb, "tickets": tickets, "plan_anual": plan_anual})

def log_loss(y, p):
    """Pérdida logarítmica media (menor es mejor)."""
    y, p = np.asarray(y), np.asarray(p)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def exactitud(y, p, umbral=0.5):
    """Proporción de aciertos al clasificar con un umbral."""
    return float(np.mean((np.asarray(p) > umbral) == np.asarray(y)))

print("Librerías y funciones cargadas correctamente.")

## Ejercicio 1 — Multicolinealidad: VIF y qué variable quitar

Con `datos` (1 500 clientes) calcula el VIF de las cinco variables explicativas (con constante, sin incluir la constante en el resultado): `vifs` es un diccionario `{variable: VIF}`. Decide qué variable quitar del par correlacionado (`var_a_quitar`: por sentido de negocio, quita la que no es la facturación) y reajusta: `se_con` es el error estándar de `gasto` en el modelo con las cinco variables y `se_sin` en el modelo sin la variable quitada.

In [ ]:
datos = generar_clientes(1500, np.random.default_rng(2026))
columnas = ["antiguedad", "gasto", "datos_gb", "tickets", "plan_anual"]
X_e1 = sm.add_constant(datos[columnas])

vifs = None            # TODO: diccionario {variable: VIF}, sin la constante
var_a_quitar = None    # TODO: nombre de la variable (texto)
se_con = None          # TODO: error estándar de gasto con las cinco variables
se_sin = None          # TODO: error estándar de gasto sin la variable quitada

# Comprobación automática
assert all(v is not None for v in (vifs, var_a_quitar, se_con, se_sin)), "Completa las líneas marcadas con TODO"
assert set(vifs) == set(columnas), "vifs debe tener una entrada por variable, sin la constante"
ref = {c: variance_inflation_factor(X_e1.values, i) for i, c in enumerate(X_e1.columns) if c != "const"}
assert all(np.isclose(vifs[c], ref[c]) for c in columnas), "usa variance_inflation_factor(X.values, i) sobre la matriz con constante"
assert var_a_quitar == "datos_gb", "quita datos_gb: gasto es la facturación, que la empresa controla"
ref_con = smf.logit("abandona ~ antiguedad + gasto + datos_gb + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]
ref_sin = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]
assert np.isclose(se_con, ref_con) and np.isclose(se_sin, ref_sin), "ajusta los dos modelos con smf.logit y toma .bse['gasto']"
assert se_sin < se_con / 2, "al quitar la variable redundante el error estándar debería bajar mucho"
print({k: round(v, 1) for k, v in vifs.items()})
print(f"Error estándar de gasto: {se_con:.4f} → {se_sin:.4f}")
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — AIC y BIC a mano entre modelos candidatos

Con 500 clientes (`datos2`) y tres modelos candidatos ya ajustados (`ajustes2`), calcula **a mano** con la log-verosimilitud `m.llf`, el número de parámetros `k = len(m.params)` y `n = m.nobs`: `aic_mano` (`2k − 2·LL`) y `bic_mano` (`k·ln(n) − 2·LL`), dos diccionarios `{nombre: valor}`. Después `elegido_aic` y `elegido_bic` son el nombre del modelo con menor valor en cada criterio, y `dif_irrelevante` es `True` si M2 y M3 se diferencian en menos de 2 puntos de AIC.

In [ ]:
datos2 = generar_clientes(500, np.random.default_rng(100))
formulas2 = {"M2": "abandona ~ antiguedad + tickets + plan_anual",
             "M3": "abandona ~ antiguedad + tickets + plan_anual + gasto",
             "M4": "abandona ~ antiguedad + tickets + plan_anual + gasto + datos_gb"}
ajustes2 = {nombre: smf.logit(f, data=datos2).fit(disp=0) for nombre, f in formulas2.items()}

aic_mano = None         # TODO: dict {nombre: 2k - 2LL}
bic_mano = None         # TODO: dict {nombre: k·ln(n) - 2LL}
elegido_aic = None      # TODO
elegido_bic = None      # TODO
dif_irrelevante = None  # TODO

# Comprobación automática
assert all(v is not None for v in (aic_mano, bic_mano, elegido_aic, elegido_bic, dif_irrelevante)), "Completa las líneas marcadas con TODO"
assert all(np.isclose(aic_mano[k], ajustes2[k].aic) for k in formulas2), "AIC = 2k - 2LL, con k = len(m.params) (incluye la constante)"
assert all(np.isclose(bic_mano[k], ajustes2[k].bic) for k in formulas2), "BIC = k·ln(n) - 2LL, con n = m.nobs"
assert elegido_aic == min(ajustes2, key=lambda k: ajustes2[k].aic) and elegido_bic == min(ajustes2, key=lambda k: ajustes2[k].bic), "elige el nombre con el menor valor de cada criterio"
assert elegido_aic == "M3" and elegido_bic == "M2", "en esta muestra el AIC prefiere M3 y el BIC prefiere el más simple, M2"
assert dif_irrelevante == True, "AIC de M2 y M3 se diferencian en menos de 2 puntos"
print("AIC:", {k: round(v, 1) for k, v in aic_mano.items()}, "→", elegido_aic)
print("BIC:", {k: round(v, 1) for k, v in bic_mano.items()}, "→", elegido_bic)
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Hold-out a mano: entrenamiento, prueba y sobreajuste

Con 300 clientes y **15 variables de ruido** añadidas, separa el 70 % para entrenamiento (`train`) y el 30 % para prueba (`test`) usando el orden aleatorio `orden` ya calculado. Ajusta el modelo con las 4 variables reales más el ruido **solo con `train`** y calcula, con `log_loss` de la Celda 0: `ll_train`, `ll_test` y `ll_base` (log-loss en `test` de la predicción constante con la tasa de abandono de `train`). Después, `hay_sobreajuste` es `True` si el log-loss en prueba es peor (mayor) que en entrenamiento.

In [ ]:
rng_e3 = np.random.default_rng(77)
datos3 = generar_clientes(300, rng_e3)
for j in range(15):
    datos3[f"nz{j + 1}"] = rng_e3.normal(size=300)
formula3 = "abandona ~ antiguedad + gasto + tickets + plan_anual" + "".join(f" + nz{j + 1}" for j in range(15))
orden = rng_e3.permutation(300)

train = None            # TODO: primeras 210 filas según orden (datos3.iloc[orden[:210]])
test = None             # TODO: las 90 restantes
ll_train = None         # TODO
ll_test = None          # TODO
ll_base = None          # TODO
hay_sobreajuste = None  # TODO

# Comprobación automática
assert all(v is not None for v in (train, test, ll_train, ll_test, ll_base, hay_sobreajuste)), "Completa las líneas marcadas con TODO"
assert len(train) == 210 and len(test) == 90, "70 % para entrenamiento (210) y 30 % para prueba (90)"
assert set(train.index).isdisjoint(test.index), "entrenamiento y prueba no deben compartir filas"
m3_ref = smf.logit(formula3, data=datos3.iloc[orden[:210]]).fit(disp=0, maxiter=200)
assert np.isclose(ll_train, log_loss(datos3.iloc[orden[:210]]["abandona"], m3_ref.predict(datos3.iloc[orden[:210]]))), "ll_train: log_loss(train['abandona'], modelo.predict(train))"
assert np.isclose(ll_test, log_loss(datos3.iloc[orden[210:]]["abandona"], m3_ref.predict(datos3.iloc[orden[210:]]))), "el modelo se ajusta solo con train y se evalúa en test"
tasa_tr = datos3.iloc[orden[:210]]["abandona"].mean()
assert np.isclose(ll_base, log_loss(datos3.iloc[orden[210:]]["abandona"], np.full(90, tasa_tr))), "la referencia usa la tasa de abandono de train, no la de test"
assert hay_sobreajuste == (ll_test > ll_train), "compara ll_test con ll_train"
print(f"log-loss: train {ll_train:.3f}   test {ll_test:.3f}   referencia constante {ll_base:.3f}   ¿sobreajuste? {hay_sobreajuste}")
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Validación cruzada 5-fold a mano

Con 300 clientes (`datos4`) y el orden aleatorio `orden4`, construye: `folds` (una lista de 5 arrays de índices, con `np.array_split(orden4, 5)`), `perdidas` (lista con el log-loss de cada bloque: el modelo `formula4` se ajusta con los **otros cuatro bloques** y se evalúa en el bloque de prueba; usa un bucle `for i in range(5)` y `np.concatenate`), `media_cv` (media de las 5 pérdidas) y `rango_bloques` (máximo menos mínimo de las 5 pérdidas).

In [ ]:
datos4 = generar_clientes(300, np.random.default_rng(5))
formula4 = "abandona ~ antiguedad + gasto + tickets + plan_anual"
orden4 = np.random.default_rng(1).permutation(300)

folds = None            # TODO: np.array_split(orden4, 5)
perdidas = None         # TODO: lista con el log-loss de cada bloque (bucle for)
media_cv = None         # TODO
rango_bloques = None    # TODO

# Comprobación automática
assert all(v is not None for v in (folds, perdidas, media_cv, rango_bloques)), "Completa las líneas marcadas con TODO"
assert len(folds) == 5 and all(len(f) == 60 for f in folds), "5 bloques de 60 clientes cada uno"
assert sorted(np.concatenate(folds)) == list(range(300)), "cada cliente debe estar en un solo bloque"
ref_p = []
for i in range(5):
    te_r = datos4.iloc[orden4[i * 60:(i + 1) * 60]]
    tr_r = datos4.iloc[np.concatenate([orden4[j * 60:(j + 1) * 60] for j in range(5) if j != i])]
    ref_p.append(log_loss(te_r["abandona"], smf.logit(formula4, data=tr_r).fit(disp=0).predict(te_r)))
assert len(perdidas) == 5 and np.allclose(perdidas, ref_p), "cada pérdida: ajusta con los otros 4 bloques y evalúa en el bloque de prueba"
assert np.isclose(media_cv, np.mean(ref_p)), "media_cv = media de las 5 pérdidas"
assert np.isclose(rango_bloques, max(ref_p) - min(ref_p)), "rango_bloques = máximo - mínimo"
print("Log-loss por bloque:", np.round(perdidas, 3), " media:", round(media_cv, 3), " rango:", round(rango_bloques, 3))
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Exactitud frente a la predicción trivial y matriz de confusión

Un modelo logístico (`m5`, ajustado con 700 clientes) predice probabilidades `p5` para 300 clientes de prueba, de los que solo un 12 % abandona (`y_test5`). Clasifica "abandona" si `p5 > 0.5` y calcula, **a mano** (con `numpy`): `vp`, `fp`, `fn`, `vn` (enteros), `exactitud5`, `precision5`, `recall5`, `exactitud_base5` (la de la predicción "nadie abandona") y `supera_base` (`True` si `exactitud5 > exactitud_base5`). Después repite con el umbral 0.2: `exactitud_02` y `supera_base_02`.

In [ ]:
rng_e5 = np.random.default_rng(55)
n5 = 1000
x1, x2 = rng_e5.normal(size=n5), rng_e5.normal(size=n5)
y5 = rng_e5.binomial(1, 1 / (1 + np.exp(-(-2.8 + 1.6 * x1 + 0.8 * x2))))
X5 = sm.add_constant(np.column_stack([x1, x2]))
m5 = sm.Logit(y5[:700], X5[:700]).fit(disp=0)
p5, y_test5 = m5.predict(X5[700:]), y5[700:]

vp = None                # TODO: predice 1 y es 1
fp = None                # TODO
fn = None                # TODO
vn = None                # TODO
exactitud5 = None        # TODO
precision5 = None        # TODO
recall5 = None           # TODO
exactitud_base5 = None   # TODO
supera_base = None       # TODO
exactitud_02 = None      # TODO: exactitud con umbral 0.2
supera_base_02 = None    # TODO

# Comprobación automática
assert all(v is not None for v in (vp, fp, fn, vn, exactitud5, precision5, recall5, exactitud_base5, supera_base, exactitud_02, supera_base_02)), "Completa las líneas marcadas con TODO"
pr_ref = np.asarray(p5) > 0.5
assert (vp, fp, fn, vn) == (int(((pr_ref == 1) & (y_test5 == 1)).sum()), int(((pr_ref == 1) & (y_test5 == 0)).sum()), int(((pr_ref == 0) & (y_test5 == 1)).sum()), int(((pr_ref == 0) & (y_test5 == 0)).sum())), "VP: predice 1 y es 1; FP: predice 1 y es 0; FN: predice 0 y es 1; VN: predice 0 y es 0"
assert vp + fp + fn + vn == 300, "los cuatro números suman 300"
assert np.isclose(exactitud5, (vp + vn) / 300), "exactitud = (VP + VN) / total"
assert np.isclose(precision5, vp / (vp + fp)), "precisión = VP / (VP + FP)"
assert np.isclose(recall5, vp / (vp + fn)), "recall = VP / (VP + FN)"
assert np.isclose(exactitud_base5, np.mean(y_test5 == 0)), "'nadie abandona' acierta en todos los que no abandonan"
assert supera_base == (exactitud5 > exactitud_base5), "compara exactitud5 con exactitud_base5"
assert np.isclose(exactitud_02, np.mean((np.asarray(p5) > 0.2) == y_test5)), "repite la exactitud con p5 > 0.2"
assert supera_base_02 == (exactitud_02 > exactitud_base5) and supera_base_02 == False, "con el umbral 0.2 la exactitud queda por debajo de la predicción trivial, aunque el recall sube"
print(f"VP={vp} FP={fp} FN={fn} VN={vn}   exactitud {exactitud5:.3f} (base {exactitud_base5:.3f})   precisión {precision5:.3f}   recall {recall5:.3f}")
print("Ejercicio 5 correcto ✔")

---
## Reto final — Elegir un modelo con criterio y validarlo

Con 3 000 clientes nuevos, separa el 70 % para entrenamiento y el 30 % para prueba. Ajusta con `train_r` tres modelos candidatos (`formulas`) y calcula: `aic_por_modelo` (AIC de cada uno, en `train_r`), `ll_test_por_modelo` (log-loss de cada uno en `test_r`), `modelo_elegido` (el nombre del de menor AIC) y `mejora_vs_base` (cuánto mejora el log-loss en prueba del elegido respecto a la predicción constante, `ll_base − ll_elegido`, positivo si mejora). Escribe en `conclusion` una frase que explique tu elección (mínimo 40 caracteres).

In [ ]:
rng_r = np.random.default_rng(99)
datos_r = generar_clientes(3000, rng_r)
orden_r = rng_r.permutation(3000)
train_r, test_r = datos_r.iloc[orden_r[:2100]], datos_r.iloc[orden_r[2100:]]
formulas = {
    "A": "abandona ~ antiguedad + tickets",
    "B": "abandona ~ antiguedad + tickets + plan_anual + gasto",
    "C": "abandona ~ antiguedad + tickets + plan_anual + gasto + datos_gb"}

aic_por_modelo = None      # TODO: dict {nombre: AIC} ajustando cada fórmula con train_r
ll_test_por_modelo = None  # TODO: dict {nombre: log-loss en test}
modelo_elegido = None      # TODO
mejora_vs_base = None      # TODO
conclusion = None          # TODO

# Comprobación automática
assert all(v is not None for v in (aic_por_modelo, ll_test_por_modelo, modelo_elegido, mejora_vs_base, conclusion)), "Completa las líneas marcadas con TODO"
ref_m = {k: smf.logit(f, data=train_r).fit(disp=0) for k, f in formulas.items()}
assert all(np.isclose(aic_por_modelo[k], ref_m[k].aic) for k in formulas), "el AIC se calcula sobre train con .aic"
assert all(np.isclose(ll_test_por_modelo[k], log_loss(test_r["abandona"], ref_m[k].predict(test_r))) for k in formulas), "evalúa en test los modelos ajustados con train"
assert modelo_elegido == min(ref_m, key=lambda k: ref_m[k].aic), "elige el nombre con menor AIC"
ll_b = log_loss(test_r["abandona"], np.full(len(test_r), train_r["abandona"].mean()))
assert np.isclose(mejora_vs_base, ll_b - ll_test_por_modelo[modelo_elegido]), "mejora = log-loss constante − log-loss del modelo elegido, ambos en test"
assert mejora_vs_base > 0, "el modelo elegido debería mejorar la predicción constante"
assert isinstance(conclusion, str) and len(conclusion) >= 40, "escribe una frase de al menos 40 caracteres"
print("AIC:", {k: round(v, 1) for k, v in aic_por_modelo.items()})
print("log-loss test:", {k: round(v, 4) for k, v in ll_test_por_modelo.items()}, "elegido:", modelo_elegido)
print("Reto final correcto ✔")

---
## Cierre de la práctica

Has recorrido el ciclo de elegir y validar un modelo: detectar la multicolinealidad, comparar candidatos con AIC y BIC, evaluar con hold-out y con k-fold, y leer una matriz de confusión con clases desiguales. Continúa con la clase 13: Test A/B y diseño de experimentos.

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
datos = generar_clientes(1500, np.random.default_rng(2026))
columnas = ["antiguedad", "gasto", "datos_gb", "tickets", "plan_anual"]
X_e1 = sm.add_constant(datos[columnas])

vifs = {c: variance_inflation_factor(X_e1.values, i) for i, c in enumerate(X_e1.columns) if c != "const"}
var_a_quitar = "datos_gb"
se_con = smf.logit("abandona ~ antiguedad + gasto + datos_gb + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]
se_sin = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]

assert all(v is not None for v in (vifs, var_a_quitar, se_con, se_sin)), "Completa las líneas marcadas con TODO"
assert set(vifs) == set(columnas), "vifs debe tener una entrada por variable, sin la constante"
ref = {c: variance_inflation_factor(X_e1.values, i) for i, c in enumerate(X_e1.columns) if c != "const"}
assert all(np.isclose(vifs[c], ref[c]) for c in columnas), "usa variance_inflation_factor(X.values, i) sobre la matriz con constante"
assert var_a_quitar == "datos_gb", "quita datos_gb: gasto es la facturación, que la empresa controla"
ref_con = smf.logit("abandona ~ antiguedad + gasto + datos_gb + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]
ref_sin = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]
assert np.isclose(se_con, ref_con) and np.isclose(se_sin, ref_sin), "ajusta los dos modelos con smf.logit y toma .bse['gasto']"
assert se_sin < se_con / 2, "al quitar la variable redundante el error estándar debería bajar mucho"
print({k: round(v, 1) for k, v in vifs.items()})
print(f"Error estándar de gasto: {se_con:.4f} → {se_sin:.4f}")
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
datos2 = generar_clientes(500, np.random.default_rng(100))
formulas2 = {"M2": "abandona ~ antiguedad + tickets + plan_anual",
             "M3": "abandona ~ antiguedad + tickets + plan_anual + gasto",
             "M4": "abandona ~ antiguedad + tickets + plan_anual + gasto + datos_gb"}
ajustes2 = {nombre: smf.logit(f, data=datos2).fit(disp=0) for nombre, f in formulas2.items()}

aic_mano = {nombre: 2 * len(m.params) - 2 * m.llf for nombre, m in ajustes2.items()}
bic_mano = {nombre: len(m.params) * np.log(m.nobs) - 2 * m.llf for nombre, m in ajustes2.items()}
elegido_aic = min(aic_mano, key=aic_mano.get)
elegido_bic = min(bic_mano, key=bic_mano.get)
dif_irrelevante = bool(abs(aic_mano["M2"] - aic_mano["M3"]) < 2)

assert all(v is not None for v in (aic_mano, bic_mano, elegido_aic, elegido_bic, dif_irrelevante)), "Completa las líneas marcadas con TODO"
assert all(np.isclose(aic_mano[k], ajustes2[k].aic) for k in formulas2), "AIC = 2k - 2LL, con k = len(m.params) (incluye la constante)"
assert all(np.isclose(bic_mano[k], ajustes2[k].bic) for k in formulas2), "BIC = k·ln(n) - 2LL, con n = m.nobs"
assert elegido_aic == min(ajustes2, key=lambda k: ajustes2[k].aic) and elegido_bic == min(ajustes2, key=lambda k: ajustes2[k].bic), "elige el nombre con el menor valor de cada criterio"
assert elegido_aic == "M3" and elegido_bic == "M2", "en esta muestra el AIC prefiere M3 y el BIC prefiere el más simple, M2"
assert dif_irrelevante == True, "AIC de M2 y M3 se diferencian en menos de 2 puntos"
print("AIC:", {k: round(v, 1) for k, v in aic_mano.items()}, "→", elegido_aic)
print("BIC:", {k: round(v, 1) for k, v in bic_mano.items()}, "→", elegido_bic)
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
rng_e3 = np.random.default_rng(77)
datos3 = generar_clientes(300, rng_e3)
for j in range(15):
    datos3[f"nz{j + 1}"] = rng_e3.normal(size=300)
formula3 = "abandona ~ antiguedad + gasto + tickets + plan_anual" + "".join(f" + nz{j + 1}" for j in range(15))
orden = rng_e3.permutation(300)

train = datos3.iloc[orden[:210]]
test = datos3.iloc[orden[210:]]
m_e3 = smf.logit(formula3, data=train).fit(disp=0, maxiter=200)
ll_train = log_loss(train["abandona"], m_e3.predict(train))
ll_test = log_loss(test["abandona"], m_e3.predict(test))
ll_base = log_loss(test["abandona"], np.full(len(test), train["abandona"].mean()))
hay_sobreajuste = bool(ll_test > ll_train)

assert all(v is not None for v in (train, test, ll_train, ll_test, ll_base, hay_sobreajuste)), "Completa las líneas marcadas con TODO"
assert len(train) == 210 and len(test) == 90, "70 % para entrenamiento (210) y 30 % para prueba (90)"
assert set(train.index).isdisjoint(test.index), "entrenamiento y prueba no deben compartir filas"
m3_ref = smf.logit(formula3, data=datos3.iloc[orden[:210]]).fit(disp=0, maxiter=200)
assert np.isclose(ll_train, log_loss(datos3.iloc[orden[:210]]["abandona"], m3_ref.predict(datos3.iloc[orden[:210]]))), "ll_train: log_loss(train['abandona'], modelo.predict(train))"
assert np.isclose(ll_test, log_loss(datos3.iloc[orden[210:]]["abandona"], m3_ref.predict(datos3.iloc[orden[210:]]))), "el modelo se ajusta solo con train y se evalúa en test"
tasa_tr = datos3.iloc[orden[:210]]["abandona"].mean()
assert np.isclose(ll_base, log_loss(datos3.iloc[orden[210:]]["abandona"], np.full(90, tasa_tr))), "la referencia usa la tasa de abandono de train, no la de test"
assert hay_sobreajuste == (ll_test > ll_train), "compara ll_test con ll_train"
print(f"log-loss: train {ll_train:.3f}   test {ll_test:.3f}   referencia constante {ll_base:.3f}   ¿sobreajuste? {hay_sobreajuste}")
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
datos4 = generar_clientes(300, np.random.default_rng(5))
formula4 = "abandona ~ antiguedad + gasto + tickets + plan_anual"
orden4 = np.random.default_rng(1).permutation(300)

folds = np.array_split(orden4, 5)
perdidas = []
for i in range(5):
    test_i = datos4.iloc[folds[i]]
    train_i = datos4.iloc[np.concatenate([folds[j] for j in range(5) if j != i])]
    m_i = smf.logit(formula4, data=train_i).fit(disp=0)
    perdidas.append(log_loss(test_i["abandona"], m_i.predict(test_i)))
media_cv = float(np.mean(perdidas))
rango_bloques = float(max(perdidas) - min(perdidas))

assert all(v is not None for v in (folds, perdidas, media_cv, rango_bloques)), "Completa las líneas marcadas con TODO"
assert len(folds) == 5 and all(len(f) == 60 for f in folds), "5 bloques de 60 clientes cada uno"
assert sorted(np.concatenate(folds)) == list(range(300)), "cada cliente debe estar en un solo bloque"
ref_p = []
for i in range(5):
    te_r = datos4.iloc[orden4[i * 60:(i + 1) * 60]]
    tr_r = datos4.iloc[np.concatenate([orden4[j * 60:(j + 1) * 60] for j in range(5) if j != i])]
    ref_p.append(log_loss(te_r["abandona"], smf.logit(formula4, data=tr_r).fit(disp=0).predict(te_r)))
assert len(perdidas) == 5 and np.allclose(perdidas, ref_p), "cada pérdida: ajusta con los otros 4 bloques y evalúa en el bloque de prueba"
assert np.isclose(media_cv, np.mean(ref_p)), "media_cv = media de las 5 pérdidas"
assert np.isclose(rango_bloques, max(ref_p) - min(ref_p)), "rango_bloques = máximo - mínimo"
print("Log-loss por bloque:", np.round(perdidas, 3), " media:", round(media_cv, 3), " rango:", round(rango_bloques, 3))
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
rng_e5 = np.random.default_rng(55)
n5 = 1000
x1, x2 = rng_e5.normal(size=n5), rng_e5.normal(size=n5)
y5 = rng_e5.binomial(1, 1 / (1 + np.exp(-(-2.8 + 1.6 * x1 + 0.8 * x2))))
X5 = sm.add_constant(np.column_stack([x1, x2]))
m5 = sm.Logit(y5[:700], X5[:700]).fit(disp=0)
p5, y_test5 = m5.predict(X5[700:]), y5[700:]

pred5 = np.asarray(p5) > 0.5
vp = int(((pred5 == 1) & (y_test5 == 1)).sum())
fp = int(((pred5 == 1) & (y_test5 == 0)).sum())
fn = int(((pred5 == 0) & (y_test5 == 1)).sum())
vn = int(((pred5 == 0) & (y_test5 == 0)).sum())
exactitud5 = (vp + vn) / len(y_test5)
precision5 = vp / (vp + fp)
recall5 = vp / (vp + fn)
exactitud_base5 = float(np.mean(y_test5 == 0))
supera_base = bool(exactitud5 > exactitud_base5)
exactitud_02 = float(np.mean((np.asarray(p5) > 0.2) == y_test5))
supera_base_02 = bool(exactitud_02 > exactitud_base5)

assert all(v is not None for v in (vp, fp, fn, vn, exactitud5, precision5, recall5, exactitud_base5, supera_base, exactitud_02, supera_base_02)), "Completa las líneas marcadas con TODO"
pr_ref = np.asarray(p5) > 0.5
assert (vp, fp, fn, vn) == (int(((pr_ref == 1) & (y_test5 == 1)).sum()), int(((pr_ref == 1) & (y_test5 == 0)).sum()), int(((pr_ref == 0) & (y_test5 == 1)).sum()), int(((pr_ref == 0) & (y_test5 == 0)).sum())), "VP: predice 1 y es 1; FP: predice 1 y es 0; FN: predice 0 y es 1; VN: predice 0 y es 0"
assert vp + fp + fn + vn == 300, "los cuatro números suman 300"
assert np.isclose(exactitud5, (vp + vn) / 300), "exactitud = (VP + VN) / total"
assert np.isclose(precision5, vp / (vp + fp)), "precisión = VP / (VP + FP)"
assert np.isclose(recall5, vp / (vp + fn)), "recall = VP / (VP + FN)"
assert np.isclose(exactitud_base5, np.mean(y_test5 == 0)), "'nadie abandona' acierta en todos los que no abandonan"
assert supera_base == (exactitud5 > exactitud_base5), "compara exactitud5 con exactitud_base5"
assert np.isclose(exactitud_02, np.mean((np.asarray(p5) > 0.2) == y_test5)), "repite la exactitud con p5 > 0.2"
assert supera_base_02 == (exactitud_02 > exactitud_base5) and supera_base_02 == False, "con el umbral 0.2 la exactitud queda por debajo de la predicción trivial, aunque el recall sube"
print(f"VP={vp} FP={fp} FN={fn} VN={vn}   exactitud {exactitud5:.3f} (base {exactitud_base5:.3f})   precisión {precision5:.3f}   recall {recall5:.3f}")
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
rng_r = np.random.default_rng(99)
datos_r = generar_clientes(3000, rng_r)
orden_r = rng_r.permutation(3000)
train_r, test_r = datos_r.iloc[orden_r[:2100]], datos_r.iloc[orden_r[2100:]]
formulas = {
    "A": "abandona ~ antiguedad + tickets",
    "B": "abandona ~ antiguedad + tickets + plan_anual + gasto",
    "C": "abandona ~ antiguedad + tickets + plan_anual + gasto + datos_gb"}

ajustes = {k: smf.logit(f, data=train_r).fit(disp=0) for k, f in formulas.items()}
aic_por_modelo = {k: m.aic for k, m in ajustes.items()}
ll_test_por_modelo = {k: log_loss(test_r["abandona"], m.predict(test_r)) for k, m in ajustes.items()}
modelo_elegido = min(aic_por_modelo, key=aic_por_modelo.get)
ll_base_r = log_loss(test_r["abandona"], np.full(len(test_r), train_r["abandona"].mean()))
mejora_vs_base = ll_base_r - ll_test_por_modelo[modelo_elegido]
conclusion = "Elijo el modelo de menor AIC, que además tiene sentido de negocio y evito la variable redundante datos_gb; lo valido en la prueba."

assert all(v is not None for v in (aic_por_modelo, ll_test_por_modelo, modelo_elegido, mejora_vs_base, conclusion)), "Completa las líneas marcadas con TODO"
ref_m = {k: smf.logit(f, data=train_r).fit(disp=0) for k, f in formulas.items()}
assert all(np.isclose(aic_por_modelo[k], ref_m[k].aic) for k in formulas), "el AIC se calcula sobre train con .aic"
assert all(np.isclose(ll_test_por_modelo[k], log_loss(test_r["abandona"], ref_m[k].predict(test_r))) for k in formulas), "evalúa en test los modelos ajustados con train"
assert modelo_elegido == min(ref_m, key=lambda k: ref_m[k].aic), "elige el nombre con menor AIC"
ll_b = log_loss(test_r["abandona"], np.full(len(test_r), train_r["abandona"].mean()))
assert np.isclose(mejora_vs_base, ll_b - ll_test_por_modelo[modelo_elegido]), "mejora = log-loss constante − log-loss del modelo elegido, ambos en test"
assert mejora_vs_base > 0, "el modelo elegido debería mejorar la predicción constante"
assert isinstance(conclusion, str) and len(conclusion) >= 40, "escribe una frase de al menos 40 caracteres"
print("AIC:", {k: round(v, 1) for k, v in aic_por_modelo.items()})
print("log-loss test:", {k: round(v, 4) for k, v in ll_test_por_modelo.items()}, "elegido:", modelo_elegido)
print("Reto final correcto ✔")

---
## Autoevaluación

Anota en una celda de texto o en papel:

- **¿Qué ejercicios resolví sin ayuda?**
- **¿Qué concepto sigue poco claro?** (por ejemplo: VIF, AIC frente a BIC, k-fold, precisión y recall, data leakage)
- **Tiempo dedicado:** ___ minutos
- **Una pregunta que quiero llevar a la próxima clase:** ...